# Tennis model training (Colab GPU tier)

Heavy compute runs here, not on the laptop. Flow:
1. Export the OBT locally: `python run.py --export-features --sport tennis`
2. Upload `data/exports/tennis_features.parquet` to your Drive folder `sports-edge/exports/`
3. Run this notebook on a **GPU runtime** (Runtime -> Change runtime type -> GPU)
4. Trained artifacts are written back to Drive `sports-edge/artifacts/`
5. Download them locally and run `python run.py --sport tennis --import-models`

No API keys are needed here (odds stay on the laptop).

In [ ]:
# 1. Install deps (LightGBM is preinstalled on Colab; add the heavy extras)
!pip -q install torch xgboost lightgbm polars scikit-learn pyarrow

In [ ]:
# 2. Mount Drive and locate the exported features
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = '/content/drive/MyDrive/sports-edge'
EXPORTS = os.path.join(BASE, 'exports')
ARTIFACTS = os.path.join(BASE, 'artifacts')
os.makedirs(ARTIFACTS, exist_ok=True)
FEATURES = os.path.join(EXPORTS, 'tennis_features.parquet')
assert os.path.exists(FEATURES), f'Upload tennis_features.parquet to {EXPORTS} first'

In [ ]:
# 3. Get the project code (so models/compare.py and features/pipeline.py import).
#    Either clone your repo or upload the package. Example with a git clone:
# !git clone <your-repo-url> /content/sports-edge
# %cd /content/sports-edge
#
# If you only uploaded the parquet, the comparison still needs TENNIS_FEATURE_COLS
# from features/pipeline.py — make sure the repo is on the path:
import sys; sys.path.insert(0, '/content/sports-edge')

In [ ]:
# 4. Confirm GPU is visible to PyTorch
import torch
print('CUDA available:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
# 5. Run the bake-off PER TOUR (ATP and WTA are separate models).
#    Each tour's winning portable model is saved to tennis_{tour}_moneyline.pkl,
#    and the MLP (deep-learning model) is trained and scored inside run_comparison.
from models.compare import run_comparison
tables = {t: run_comparison(features_path=FEATURES, tour=t) for t in ('atp', 'wta')}
display(tables['atp']); display(tables['wta'])

In [ ]:
# 6. Also train per-tour totals + spread regressors, then copy all artifacts
#    back to Drive for the laptop to import.
import polars as pl, shutil, glob
from models.train import train_tennis_totals, train_tennis_spread
df = pl.read_parquet(FEATURES)
for tour in ('atp', 'wta'):
    train_tennis_totals(df, tour)
    train_tennis_spread(df, tour)

for f in glob.glob('models/artifacts/tennis_*'):
    shutil.copy(f, ARTIFACTS)
print('Copied artifacts to', ARTIFACTS)